# YOLO-UDD — Option B: Pretrained YOLOv9c on TrashCan (Ultralytics)

Fine-tunes a **COCO-pretrained YOLOv9c** (the paper's baseline model) on
TrashCan 1.0, 3 classes. This is the paper-competitive baseline; TAFM is
integrated in a later step.

### Before you run
1. **Settings -> Accelerator -> GPU** (T4).
2. **Settings -> Internet -> ON** (needed to download yolov9c.pt pretrained weights).
3. **+ Add Input -> Datasets** -> add the TrashCan dataset.

Fine-tuning from pretrained converges fast (often fits in one session).

## 1. Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 2. Get the code (for the dataset converter)

In [ ]:
%cd /kaggle/working
![ -d YOLO-UDD-v2.0 ] && rm -rf YOLO-UDD-v2.0
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0


## 3. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()


## 4. Config  <- EDIT dataset path if needed

In [ ]:
DATA_DIR = '/kaggle/input/datasets/mexwell/trashcan-1-0/dataset/instance_version'
EPOCHS   = 100
BATCH    = 16      # yolov9c on a T4; lower to 8 if you hit OOM
IMGSZ    = 640
YOLO_DIR = '/kaggle/working/trashcan_yolo'   # converted dataset lives here
PROJECT  = '/kaggle/working/runs_ultra'      # training outputs
NAME     = 'yolov9c_trashcan'
import os; print('DATA_DIR exists:', os.path.isdir(DATA_DIR))


## 5. Convert TrashCan (COCO) -> YOLO format

In [ ]:
cmd = f'python scripts/coco_to_yolo_trashcan.py --dataset-dir "{DATA_DIR}" --out-dir {YOLO_DIR}'
print(cmd)
!{cmd}


## 6. Train pretrained YOLOv9c
Auto-resumes if a previous `last.pt` exists in this session. If a session ends
before 100 epochs, re-run cells 1-5 then this cell (Ultralytics resumes from
`last.pt` under PROJECT/NAME).

In [ ]:
from ultralytics import YOLO
import os
last = f'{PROJECT}/{NAME}/weights/last.pt'
if os.path.exists(last):
    print('Resuming from', last)
    model = YOLO(last)
    results = model.train(resume=True)
else:
    print('Starting from COCO-pretrained yolov9c.pt')
    model = YOLO('yolov9c.pt')
    results = model.train(
        data=f'{YOLO_DIR}/data.yaml',
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=PROJECT, name=NAME,
        patience=30, save=True, exist_ok=True,
    )


## 7. Validate -> final mAP

In [ ]:
from ultralytics import YOLO
best = f'{PROJECT}/{NAME}/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=f'{YOLO_DIR}/data.yaml', imgsz=IMGSZ)
print('mAP@50   :', round(float(metrics.box.map50), 4))
print('mAP@50:95:', round(float(metrics.box.map), 4))
print('precision:', round(float(metrics.box.mp), 4))
print('recall   :', round(float(metrics.box.mr), 4))


## 8. Save your model
Download `runs_ultra/yolov9c_trashcan/weights/best.pt` from the Output tab.
That is your trained Option-B model.